# RUNNING THIS NOTEBOOK

## MODE 1 - Local (default, no setup needed)
Uses Weaviate Embedded + sentence-transformers. Runs on CPU. Free.

Run:
`jupyter notebook`

## MODE 2 - Production (Weaviate Cloud + OpenAI)
1. Create a free cluster at console.weaviate.cloud, then copy your cluster URL and API key.
2. Get an OpenAI API key at platform.openai.com.
3. Replace Cell 1 with:

```python
client = weaviate.connect_to_weaviate_cloud(
    cluster_url="YOUR_WEAVIATE_URL",
    auth_credentials=weaviate.auth.AuthApiKey("YOUR_WEAVIATE_API_KEY"),
    headers={"X-OpenAI-Api-Key": "YOUR_OPENAI_API_KEY"},
)
```

4. Replace collection creation in Cell 2 with:

```python
client.collections.create(
    name="EnterpriseKnowledgeBase",
    multi_tenancy_config=Configure.multi_tenancy(enabled=True),
    vectorizer_config=Configure.Vectorizer.text2vec_openai(),
    ...
)
```

5. Replace `tenant_search()` in Cell 5 with LangChain `WeaviateVectorStore` (see langchain-weaviate docs for a full example).

# Enterprise Multi-Tenant RAG with Weaviate + LangChain

This notebook demonstrates a **production-style multi-tenant RAG system** using Weaviate.

Key features:
- Tenant isolation
- Hybrid search (vector + BM25)
- Local embeddings (no API keys)
- LangChain-style retrieval

## 1. Setup Weaviate (Local Embedded) + Create Collection

In [ ]:
import weaviate
from weaviate.embedded import EmbeddedOptions

client = weaviate.WeaviateClient(
    embedded_options=EmbeddedOptions()
)

client.connect()
print("Connected:", client.is_ready())

from weaviate.classes.config import Configure, Property, DataType

client.collections.delete("EnterpriseKnowledgeBase")

client.collections.create(
    name="EnterpriseKnowledgeBase",
    multi_tenancy_config=Configure.multi_tenancy(enabled=True),
    properties=[
        Property(name="content", data_type=DataType.TEXT),
        Property(name="source", data_type=DataType.TEXT),
        Property(name="department", data_type=DataType.TEXT),
    ]
)

print("Collection created")

## 2. Create Tenants (Isolation Layer)

In [ ]:
from weaviate.classes.tenants import Tenant

collection = client.collections.get("EnterpriseKnowledgeBase")
existing = collection.tenants.get()

if not existing:
    collection.tenants.create([
        Tenant(name="engineering"),
        Tenant(name="finance"),
        Tenant(name="legal"),
    ])

print("Tenants ensured")

## 3. Local Embeddings (No API Key Needed)

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

def embed(text):
    return model.encode(text).tolist()

## 4. Ingest Documents per Tenant

Each tenant gets completely isolated data.

In [ ]:
engineering_docs = [
    {"content": "RAG uses semantic chunking.", "source": "eng", "department": "engineering"},
    {"content": "Pipeline serves 5M predictions/day.", "source": "eng", "department": "engineering"},
]

finance_docs = [
    {"content": "Budget allocated $2.4M.", "source": "fin", "department": "finance"},
    {"content": "Cloud cost reduced 38%.", "source": "fin", "department": "finance"},
]

eng = collection.with_tenant("engineering")
fin = collection.with_tenant("finance")

with eng.batch.dynamic() as batch:
    for doc in engineering_docs:
        batch.add_object(
            properties=doc,
            vector=embed(doc["content"])
        )

with fin.batch.dynamic() as batch:
    for doc in finance_docs:
        batch.add_object(
            properties=doc,
            vector=embed(doc["content"])
        )

print("Data ingested")

## 5. Hybrid Search (Vector + Keyword)

This is critical for production RAG.

In [ ]:
def tenant_search(tenant_name, query, top_k=2):
    col = client.collections.get("EnterpriseKnowledgeBase")
    tenant_col = col.with_tenant(tenant_name)

    query_vec = embed(query)

    results = tenant_col.query.hybrid(
        query=query,
        vector=query_vec,
        alpha=0.5,
        limit=top_k,
        return_properties=["content", "source", "department"]
    )

    if not results.objects:
        print(f"No results for tenant: {tenant_name}")
        return []

    return results.objects

results = tenant_search("engineering", "RAG system")
for r in results:
    print(r.properties["content"])

## 6. Metadata Filtering (Enterprise Control Layer)

In [ ]:
from weaviate.classes.query import Filter

def filtered_search(tenant_name, query):
    col = client.collections.get("EnterpriseKnowledgeBase")
    tenant_col = col.with_tenant(tenant_name)

    results = tenant_col.query.hybrid(
        query=query,
        vector=embed(query),
        alpha=0.5,
        filters=Filter.by_property("department").equal(tenant_name),
        limit=2,
        return_properties=["content"]
    )

    return results.objects

results = filtered_search("engineering", "pipeline")
for r in results:
    print(r.properties["content"])

## 7. LangChain-style Retriever (Minimal)

In [ ]:
from langchain_core.documents import Document

def langchain_retriever(tenant, query):
    results = tenant_search(tenant, query)

    return [
        Document(page_content=r.properties["content"], metadata=r.properties)
        for r in results
    ]

docs = langchain_retriever("engineering", "RAG")
for d in docs:
    print(d.page_content)